# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [25]:
import numpy as np 
import pandas as pd
import os

import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv('../cred.env')

# retrieve API keys from .env file

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use. 

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text. 

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available. 

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [4]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

#news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
#MAX_NEWS = 1000
#DOCUMENT="description"
#TOPIC="title"

news = pd.read_csv('/Users/smithn5/ironhack/langchain_0_3/lab-langchain-med/datasets/articles.csv')
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"


ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [5]:
news["id"] = news.index
news.head()

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...,0
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...,1
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...,2
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...,3
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...,4


In [6]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database. 

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior. 

In [ ]:
#!pip install chromadb

In [7]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters 
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance. 

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that. 

In [9]:
Settings_chroma = Settings(chroma_db_impl="duckdb+parquet", persist_directory="./chroma_persist")
chroma_client = chromadb.PersistentClient(path="/Users/smithn5/ironhack/langchain_0_3/lab-langchain-med/chroma_persist")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it. 

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [10]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
    

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***. 
* In the **document** we store the big text, it's a different column in each Dataset. 
* In **metadatas**, we can informa a list of topics. 
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS. 


In [11]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

/Users/smithn5/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:43<00:00, 1.93MiB/s]


In [12]:
results = collection.query(query_texts=["laptop"], n_results=100)

print(results)

{'ids': [['id9', 'id85', 'id1', 'id80', 'id52', 'id27', 'id73', 'id93', 'id6', 'id98']], 'embeddings': None, 'documents': [['[\'How will advances in computing transform human society?\', \'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.\', \'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.\', \'Students dreamed u

## Vector MAP

In [14]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [17]:

getado = collection.get(ids="id98", 
                       include=["documents", "embeddings"])


In [18]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[ 1.64806377e-02, -1.05098195e-01,  4.12227400e-02,
        -5.36262579e-02, -3.33806057e-03,  1.62998363e-02,
        -1.88527291e-03, -2.60295272e-02,  1.11483308e-02,
        -1.75798144e-02, -7.94970766e-02, -5.01478054e-02,
        -1.61884297e-02,  4.76559810e-03,  1.39789665e-02,
         1.71011582e-03,  1.04735397e-01,  7.47525394e-02,
        -3.85922007e-02, -8.72070417e-02,  3.61388065e-02,
         2.71336399e-02,  3.50330472e-02, -5.48493117e-04,
        -1.01981256e-02,  3.87011357e-02,  3.91494483e-02,
        -6.17149808e-02, -1.51279732e-03, -2.74985214e-03,
         5.39355278e-02,  2.70635448e-02,  7.59760439e-02,
        -3.14551927e-02, -8.66764858e-02, -5.79602597e-03,
         5.92005514e-02, -9.58840828e-03,  2.63297018e-02,
        -2.09694654e-02, -7.04626217e-02, -3.14992368e-02,
         2.51802243e-02, -4.99141216e-02,  4.08012643e-02,
        -2.77277455e-02, -4.69609648e-02, -8.68649781e-02,
        -4.94138710e-02,  3.25066932e-02, -9.72862765e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents. 

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search. 


## Loading the model and creating the prompt
Time to use the OpenAI API for text generation with **gpt-4o-mini**.

We will:
* Create an OpenAI client using your `OPENAI_API_KEY`.
* Set the model to `gpt-4o-mini`.
* Send the prompt built from Chroma context and the user question.

In [27]:
# !pip install openai
import os
from openai import OpenAI



model_id = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

We no longer need a local Hugging Face pipeline.

We will configure generation parameters for OpenAI and keep responses concise for this project.

In [28]:
max_output_tokens = 256
temperature = 0.2

## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user. 

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**. 

We only need to join the two parts together to create the prompt that we are going to send to the model. 

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part. 

In [31]:
question = "Any grants in AI where handed?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

'Relevant context: #[\'How will advances in computing transform human society?\', \'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.\', \'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.\', \'Students dreamed up highly inventive scenarios for how the technologies of today and tomorrow could impact society, for bet

Now all that remains is to send the prompt to the model and wait for its response!


In [32]:
response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant that answers using the provided context."},
        {"role": "user", "content": prompt_template},
    ],
    max_tokens=max_output_tokens,
    temperature=temperature,
    )

print(response.choices[0].message.content)

The context provided does not mention any grants specifically related to AI being handed out. However, it does discuss an essay contest called the Envisioning the Future of Computing Prize, where the winners received monetary awards for their essays on the implications of computing technologies, including AI. Robert Cunningham won the grand prize of $10,000, while the other two finalists received $5,000 each.
